# 🌱 EcoRiego AI — Notebook 1: Generación de Datos Sintéticos Realistas
**Proyecto:** EcoRiego AI — Sistema de Riego Autónomo con TinyML para ESP32  
**Curso:** Tópicos Especiales en Sistemas Inteligentes | UNC Cajamarca

---
### Flujo de este Notebook (Pipeline MLOps)
```
FASE 1: Configuración del Entorno Agrícola (Cultivos, Parámetros)
    ↓
FASE 2: Funciones de Física Real (Ciclo Circadiano, ET Penman-Monteith)
    ↓
FASE 3: Lógica Maestra Agronómica (Evapotranspiración + Reglas de Campo)
    ↓
FASE 4: Generación de 20,000 Registros Correlacionados
    ↓
FASE 5: Validación Estadística y Visualización
    ↓
FASE 6: Exportación del Dataset (CSV)
```


## FASE 1 — Configuración del Entorno Agrícola

In [ ]:
# ── Instalación (ejecutar solo si es necesario) ──
# !pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

print("✅ Librerías cargadas.")
print(f"   NumPy v{np.__version__}  |  Pandas v{pd.__version__}")


In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURACIÓN MAESTRA — Edita aquí para escalar el sistema
# ══════════════════════════════════════════════════════════

CANTIDAD_DATOS    = 20_000
SEMILLA_ALEATORIA = 42
np.random.seed(SEMILLA_ALEATORIA)

# ── Perfiles de Cultivo (escalable: añade más cultivos aquí) ──
# kc = Coeficiente de Cultivo FAO-56 (demanda evapotranspirativa)
CULTIVOS = {
    "tomate":  {"umbral_riego": 45, "umbral_optimo": 65, "kc": 1.05},
    "lechuga": {"umbral_riego": 55, "umbral_optimo": 70, "kc": 0.90},
    "papa":    {"umbral_riego": 40, "umbral_optimo": 60, "kc": 1.10},
}

# ── Fases de Crecimiento (factor multiplicador de demanda hídrica) ──
FASES_CRECIMIENTO = {
    "germinacion": 0.50,
    "vegetativa":  1.00,
    "floracion":   1.20,
    "maduracion":  0.80,
}

print("=" * 55)
print("  EcoRiego AI — Configuración del Entorno Agrícola")
print("  Contexto: Invernadero serrano (Cajamarca ~2700 msnm)")
print("=" * 55)
print(f"  Registros a generar : {CANTIDAD_DATOS:,}")
print(f"  Cultivos disponibles: {list(CULTIVOS.keys())}")
print(f"  Fases de crecimiento: {list(FASES_CRECIMIENTO.keys())}")
print("=" * 55)


## FASE 2 — Funciones de Física Real
Estas funciones modelan el comportamiento real del invernadero:
- **Temperatura**: Sigue un ciclo circadiano (pico a las 14h, mínimo al amanecer)
- **Humedad del aire**: Inversamente correlacionada con temperatura (real: a más calor, aire más seco)
- **Evapotranspiración**: Modelo Penman-Monteith simplificado (FAO-56), usado globalmente en agronomía


In [ ]:
def generar_temperatura_circadiana(hora_dia: float, mes: int) -> float:
    """Temperatura del invernadero según hora y mes (°C). Cajamarca, Perú."""
    temp_base  = 18.0
    amplitud   = 10.0
    ajuste_mes = 4.0 if 4 <= mes <= 10 else -2.0  # Época seca vs lluviosa
    var_hora   = amplitud * math.sin(math.pi * (hora_dia - 6) / 12) if 6 <= hora_dia <= 18 else -amplitud * 0.5
    temp = temp_base + ajuste_mes + var_hora + np.random.normal(0, 1.5)
    return float(np.clip(temp, 8.0, 38.0))

def generar_humedad_aire(temperatura: float, mes: int) -> float:
    """Humedad relativa del aire (%). Inversamente correlacionada con temperatura."""
    hum_base  = 75.0 if mes in [11, 12, 1, 2, 3] else 55.0  # Más húmedo en época lluviosa
    ajuste_t  = -1.5 * max(0, temperatura - 20)  # Cada °C extra sobre 20 reduce 1.5% HR
    hum = hum_base + ajuste_t + np.random.normal(0, 5.0)
    return float(np.clip(hum, 20.0, 98.0))

def calcular_et_referencia(temp: float, hum_aire: float, hora: float) -> float:
    """
    Tasa de evapotranspiración (mm/hora) — Penman-Monteith simplificado (FAO-56).
    ET alta = planta evapora más = necesita más agua.
    """
    factor_solar = max(0, math.sin(math.pi * (hora - 6) / 12)) if 6 <= hora <= 18 else 0.0
    # VPD (Vapor Pressure Deficit): motor real de la transpiración
    vpd = (1 - hum_aire / 100.0) * 0.611 * math.exp(17.27 * temp / (temp + 237.3))
    et  = 0.408 * factor_solar * 0.5 + 0.066 * vpd * 2.0
    return float(np.clip(et, 0.0, 2.5))

# ── Prueba rápida ──
print("✅ Funciones de física agrícola definidas.")
print(f"   Temperatura a las 14h (junio): {generar_temperatura_circadiana(14, 6):.1f}°C")
print(f"   Temperatura a las 04h (enero): {generar_temperatura_circadiana(4, 1):.1f}°C")
print(f"   Humedad aire con 30°C        : {generar_humedad_aire(30, 7):.1f}%")
print(f"   ET al mediodía (25°C, 50%HR) : {calcular_et_referencia(25, 50, 12):.3f} mm/h")


## FASE 3 — Lógica Maestra Agronómica
**Diferencia clave** con la versión anterior: esta función incorpora conocimiento agronómico real:

| Regla | Lógica | Fundamento |
|-------|--------|-----------|
| R1 | No regar si suelo ≥ umbral_óptimo | Evita encharcamiento y hongos radiculares |
| R2 | No regar de noche (23h-6h) | Previene Botrytis y mohos en invernadero |
| R3 | No regar con niebla + frío | Alta HR ambiental → planta absorbe agua del aire |
| R4 | Ajuste por ET real | A más evapotranspiración → más tiempo de riego |
| R5 | Ajuste por temperatura extrema | Calor → +20% agua; Frío → -25% agua |
| R6 | Ruido realista | Variación de presión de bomba y caudal |


In [ ]:
def calcular_tiempo_riego_maestro(
    humedad_suelo: float,
    temperatura: float,
    humedad_aire: float,
    hora_dia: float,
    cultivo: str,
    fase: str
) -> float:
    """
    Calcula el tiempo de riego óptimo en minutos según condiciones reales.
    Retorna 0.0 si no se requiere riego.
    """
    perfil    = CULTIVOS[cultivo]
    factor_et = FASES_CRECIMIENTO[fase]

    # R1: No regar si suelo suficientemente húmedo
    if humedad_suelo >= perfil["umbral_optimo"]:
        return 0.0

    # R2: No regar de noche (riesgo de hongos en invernadero)
    if hora_dia < 6 or hora_dia >= 23:
        return 0.0

    # R3: No regar si hay niebla y temperatura baja
    if humedad_aire > 88.0 and temperatura < 18.0:
        return 0.0

    # Cálculo del déficit de humedad
    deficit = perfil["umbral_riego"] - humedad_suelo
    if deficit <= 0:
        return 0.0

    # Tiempo base proporcional al déficit × coeficiente de cultivo
    tiempo_base = deficit * 0.12 * perfil["kc"]

    # R4: Ajuste por Evapotranspiración real
    et = calcular_et_referencia(temperatura, humedad_aire, hora_dia)
    ajuste_et = et * factor_et * 1.5

    tiempo = tiempo_base + ajuste_et

    # R5: Ajuste por temperatura extrema
    if temperatura > 30.0:
        tiempo *= 1.20   # Calor: más evaporación → más riego
    elif temperatura < 12.0:
        tiempo *= 0.75   # Frío: menos evaporación → menos riego

    # Ajuste por aire muy seco
    if humedad_aire < 35.0:
        tiempo *= 1.15

    # Límites físicos de la bomba (0 a 15 minutos)
    tiempo = np.clip(tiempo, 0.0, 15.0)

    # R6: Ruido realista (variación de presión/caudal de bomba)
    tiempo += np.random.normal(0, 0.15)
    return max(0.0, round(float(tiempo), 2))

print("✅ Lógica Maestra Agronómica definida. Prueba:")
print(f"   Suelo=10% Temp=28°C HumAire=40% 14h tomate vegetativa → {calcular_tiempo_riego_maestro(10,28,40,14,'tomate','vegetativa')} min")
print(f"   Suelo=70% Temp=20°C HumAire=60% 10h tomate vegetativa → {calcular_tiempo_riego_maestro(70,20,60,10,'tomate','vegetativa')} min (debería ser 0)")
print(f"   Suelo=30% Temp=10°C HumAire=75% 03h tomate vegetativa → {calcular_tiempo_riego_maestro(30,10,75,3,'tomate','vegetativa')} min (noche: debería ser 0)")


## FASE 4 — Generación de 20,000 Registros

In [ ]:
print(f"⏳ Generando {CANTIDAD_DATOS:,} registros con física real...")

lista_cultivos = list(CULTIVOS.keys())
lista_fases    = list(FASES_CRECIMIENTO.keys())
registros = []

for i in range(CANTIDAD_DATOS):
    # Variables contextuales (influyen en la lógica, no van al modelo)
    mes      = np.random.randint(1, 13)
    hora_dia = round(np.random.uniform(0, 23.99), 2)
    cultivo  = np.random.choice(lista_cultivos)
    fase     = np.random.choice(lista_fases, p=[0.10, 0.40, 0.30, 0.20])

    # Variables de sensor (ENTRADAS al modelo)
    temperatura  = generar_temperatura_circadiana(hora_dia, mes)
    humedad_aire = generar_humedad_aire(temperatura, mes)

    # Humedad del suelo: distribución bimodal (campo real tiene suelos secos O húmedos)
    if np.random.random() < 0.40:
        humedad_suelo = float(np.random.beta(2, 5) * 55)        # Tendencia seco
    else:
        humedad_suelo = float(np.random.beta(5, 2) * 55 + 45)   # Tendencia húmedo
    humedad_suelo = float(np.clip(round(humedad_suelo, 2), 0.0, 100.0))

    # ETIQUETA (SALIDA del modelo)
    tiempo_riego = calcular_tiempo_riego_maestro(
        humedad_suelo, temperatura, humedad_aire, hora_dia, cultivo, fase
    )

    registros.append({
        "HumedadSuelo":    round(humedad_suelo, 2),
        "TemperaturaAire": round(temperatura, 2),
        "HumedadAire":     round(humedad_aire, 2),
        "TiempoRiego":     round(tiempo_riego, 2),
        # Metadata de contexto (solo para análisis)
        "_HoraDia": hora_dia, "_Mes": mes,
        "_Cultivo": cultivo,  "_Fase": fase,
    })

    if (i + 1) % 5000 == 0:
        print(f"   {i+1:,}/{CANTIDAD_DATOS:,} completados...")

df = pd.DataFrame(registros)
print(f"\n✅ Dataset generado: {len(df):,} filas × {len(df.columns)} columnas")


## FASE 5 — Validación Estadística y Visualización

In [ ]:
print("=" * 60)
print("  VALIDACIÓN DEL DATASET")
print("=" * 60)

print("\n📊 Estadísticas descriptivas (features del modelo):")
print(df[["HumedadSuelo", "TemperaturaAire", "HumedadAire", "TiempoRiego"]].describe().round(2))

casos_sin_riego = (df["TiempoRiego"] == 0).sum()
casos_con_riego = (df["TiempoRiego"] > 0).sum()
print(f"\n🌱 Distribución de riego:")
print(f"   Sin riego (0 min)  : {casos_sin_riego:,} ({casos_sin_riego/len(df)*100:.1f}%)")
print(f"   Con riego (>0 min) : {casos_con_riego:,} ({casos_con_riego/len(df)*100:.1f}%)")

print(f"\n🔗 Correlaciones con TiempoRiego:")
corr = df[["HumedadSuelo","TemperaturaAire","HumedadAire","TiempoRiego"]].corr()["TiempoRiego"].drop("TiempoRiego")
for k, v in corr.items():
    bar = "█" * int(abs(v)*20)
    signo = "+" if v > 0 else "-"
    print(f"   {k:<18}: {signo}{abs(v):.3f}  {bar}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("EcoRiego AI — Análisis del Dataset Sintético Realista", fontsize=14, fontweight='bold')

# 1. Humedad del Suelo (distribución bimodal)
axes[0,0].hist(df["HumedadSuelo"], bins=40, color="#4CAF50", edgecolor='white', alpha=0.85)
axes[0,0].axvline(50, color='red', linestyle='--', linewidth=1.5, label='Umbral ~50%')
axes[0,0].set_title("Distribución: Humedad del Suelo")
axes[0,0].set_xlabel("Humedad (%)"); axes[0,0].set_ylabel("Frecuencia")
axes[0,0].legend()

# 2. Temperatura (ciclo circadiano)
axes[0,1].hist(df["TemperaturaAire"], bins=40, color="#FF9800", edgecolor='white', alpha=0.85)
axes[0,1].set_title("Distribución: Temperatura del Aire")
axes[0,1].set_xlabel("Temperatura (°C)"); axes[0,1].set_ylabel("Frecuencia")

# 3. Tiempo de Riego (solo casos con riego)
axes[0,2].hist(df[df["TiempoRiego"]>0]["TiempoRiego"], bins=40, color="#2196F3", edgecolor='white', alpha=0.85)
axes[0,2].set_title("Distribución: Tiempo de Riego (casos activos)")
axes[0,2].set_xlabel("Minutos"); axes[0,2].set_ylabel("Frecuencia")

# 4. Suelo vs Tiempo (coloreado por temperatura)
sample = df.sample(1500, random_state=1)
sc = axes[1,0].scatter(sample["HumedadSuelo"], sample["TiempoRiego"],
                        c=sample["TemperaturaAire"], cmap="RdYlGn_r", alpha=0.4, s=8)
plt.colorbar(sc, ax=axes[1,0], label="Temp (°C)")
axes[1,0].set_title("Suelo vs Tiempo Riego (color=Temp)")
axes[1,0].set_xlabel("Humedad Suelo (%)"); axes[1,0].set_ylabel("Tiempo Riego (min)")

# 5. Temperatura vs Humedad Aire (correlación inversa)
sc2 = axes[1,1].scatter(sample["TemperaturaAire"], sample["HumedadAire"],
                         c=sample["_HoraDia"], cmap="twilight", alpha=0.4, s=8)
plt.colorbar(sc2, ax=axes[1,1], label="Hora del día")
axes[1,1].set_title("Temp vs Humedad Aire (correlación real)")
axes[1,1].set_xlabel("Temperatura (°C)"); axes[1,1].set_ylabel("Humedad Aire (%)")

# 6. Boxplot por cultivo
cultivos_names = list(CULTIVOS.keys())
cultivos_data  = [df[df["_Cultivo"]==c]["TiempoRiego"].values for c in cultivos_names]
bp = axes[1,2].boxplot(cultivos_data, labels=cultivos_names, patch_artist=True,
                        boxprops=dict(facecolor="#E8F5E9", color="#2E7D32"),
                        medianprops=dict(color="#D32F2F", linewidth=2))
axes[1,2].set_title("Tiempo de Riego por Cultivo")
axes[1,2].set_xlabel("Cultivo"); axes[1,2].set_ylabel("Tiempo (min)")

plt.tight_layout()
plt.savefig("analisis_dataset.png", dpi=120, bbox_inches='tight')
plt.show()
print("📊 Gráfico guardado: analisis_dataset.png")


## FASE 6 — Exportación del Dataset

In [ ]:
# Dataset completo con metadata (para análisis y debugging)
df.to_csv("dataset_ecoriego_completo.csv", index=False)

# Dataset reducido — SOLO FEATURES DEL MODELO (para Notebook 2)
df_modelo = df[["HumedadSuelo", "TemperaturaAire", "HumedadAire", "TiempoRiego"]].copy()
df_modelo.to_csv("dataset_riego_sprint1.csv", index=False)

print("=" * 60)
print("  ✅ EXPORTACIÓN COMPLETADA")
print("=" * 60)
print(f"  📁 dataset_riego_sprint1.csv      → {len(df_modelo):,} registros  ← PARA EL MODELO")
print(f"  📁 dataset_ecoriego_completo.csv  → Con metadata de contexto")
print(f"  📁 analisis_dataset.png           → Visualizaciones")
print()
print(f"  Con riego  : {casos_con_riego:,} ({casos_con_riego/len(df)*100:.1f}%)")
print(f"  Sin riego  : {casos_sin_riego:,} ({casos_sin_riego/len(df)*100:.1f}%)")
print()
print("  → Ejecuta el Notebook 2 para entrenar el modelo RL.")
print("=" * 60)
